# Agent 11 — Portfolio Construction

**What this notebook does:**  
Combines the financial metrics (Sharpe ratio) and ESG scores to rank all 50+ companies, apply exclusion rules, and select the final 15–25 holdings with weights.

**How to present this to investors:**  
> *Our portfolio construction agent uses a transparent composite score combining financial quality and ESG quality. We apply explicit exclusion rules, enforce diversification constraints (max 10% per stock, min 5 sectors), and the final weights are rule-based — not a black box.*

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import glob

# Load ESG scores
esg_files = sorted(glob.glob("../outputs/scores/esg_scores_*.csv"))
fin_files = sorted(glob.glob("../outputs/scores/financial_metrics_*.csv"))

if not esg_files or not fin_files:
    raise FileNotFoundError("Run notebooks 02 and 03 first.")

df_esg = pd.read_csv(esg_files[-1])
df_fin = pd.read_csv(fin_files[-1])

print(f"ESG scores: {len(df_esg)} companies")
print(f"Financial metrics: {len(df_fin)} companies")

ESG scores: 70 companies
Financial metrics: 56 companies


## Step 1 — Merge ESG and financial data

We combine both datasets on the ticker symbol.

In [2]:
# === ACTION REQUIRED ===
# Make sure both files have a 'ticker' column. Adjust if column name differs.
TICKER_COL = "ticker"

combined = df_esg.merge(df_fin, on=TICKER_COL, how="inner", suffixes=("", "_fin"))
print(f"Companies with both ESG and financial data: {len(combined)}")
combined.head(3)

Companies with both ESG and financial data: 70


,ticker,E_score,S_score,G_score,ESG_score,data_vintage,annual_return_pct,annual_volatility_pct,sharpe_ratio,max_drawdown_pct,data_vintage_fin
0,ASML.AS,76.71,51.43,34.54,56.48,2026-05-06,56.51,28.06,1.907,-23.20,2026-05-06
1,SAP.DE,89.28,50.87,56.30,67.86,2026-05-06,22.30,26.81,0.720,-48.41,2026-05-06
2,CAP.PA,78.70,53.84,41.53,60.09,2026-05-06,29.10,28.29,0.923,-25.42,2026-05-06


## Step 2 — Apply exclusions

Before scoring, we remove companies that fail our minimum standards.  
This is the greenwashing / controversy watchlist filter.

**Adjust the thresholds below to match your mandate.**

In [3]:
exclusions = []  # will collect (ticker, reason) pairs

# Rule 1: Exclude companies with very low ESG scores (bottom 10%)
ESG_FLOOR = combined["ESG_score"].quantile(0.10)
low_esg = combined[combined["ESG_score"] < ESG_FLOOR]
for _, row in low_esg.iterrows():
    exclusions.append((row[TICKER_COL], f"ESG score below floor ({row['ESG_score']:.1f} < {ESG_FLOOR:.1f})"))

# Rule 2: Exclude companies with negative Sharpe ratio (losing money per unit of risk)
neg_sharpe = combined[combined["sharpe_ratio"] < 0]
for _, row in neg_sharpe.iterrows():
    exclusions.append((row[TICKER_COL], f"Negative Sharpe ratio ({row['sharpe_ratio']:.2f})"))

# Rule 3: Manual exclusions from greenwashing checks (from Claude Projects RAG)
MANUAL_EXCLUSIONS = [
    # Add tickers here after running the greenwashing 8-Test
    # ("BP.L", "Failed greenwashing 8-Test: vague net-zero claim, no Scope 3 target"),
]
exclusions.extend(MANUAL_EXCLUSIONS)

excluded_tickers = list(set([t for t, _ in exclusions]))
eligible = combined[~combined[TICKER_COL].isin(excluded_tickers)].copy()

print(f"Companies excluded: {len(excluded_tickers)}")
print(f"Eligible for portfolio: {len(eligible)}")

# Save exclusion log
excl_df = pd.DataFrame(exclusions, columns=["ticker", "reason"]).drop_duplicates()
excl_df.to_csv("../outputs/portfolio/exclusions.csv", index=False)
print("\nExclusion log saved.")
excl_df

Companies excluded: 17
Eligible for portfolio: 53

Exclusion log saved.


,ticker,reason
0,VWS.CO,ESG score below floor (49.1 < 50.6)
1,EQNR.OL,ESG score below floor (48.5 < 50.6)
2,ORSTED.CO,ESG score below floor (43.2 < 50.6)
3,ENEL.MI,ESG score below floor (32.7 < 50.6)
4,BN.PA,ESG score below floor (46.4 < 50.6)
5,VOW3.DE,ESG score below floor (40.1 < 50.6)
6,GFC.PA,ESG score below floor (44.6 < 50.6)
7,VWS.CO,Negative Sharpe ratio (-0.05)
8,SHEL.L,Negative Sharpe ratio (-0.26)
9,IBE.MC,Negative Sharpe ratio (-0.23)


## Step 3 — Composite ranking score

We combine ESG quality and financial quality into one ranking score.  
Adjust the weights to reflect your portfolio mandate.

In [4]:
# === Composite score weights (must sum to 1.0) ===
ESG_WEIGHT_COMPOSITE   = 0.60   # 60% weight on ESG quality
SHARPE_WEIGHT_COMPOSITE = 0.40  # 40% weight on financial quality (Sharpe)

# Normalise Sharpe ratio to 0-100 for comparability
sharpe_min = eligible["sharpe_ratio"].min()
sharpe_max = eligible["sharpe_ratio"].max()
eligible["sharpe_score"] = ((eligible["sharpe_ratio"] - sharpe_min) / (sharpe_max - sharpe_min) * 100).round(2)

eligible["composite_score"] = (
    eligible["ESG_score"] * ESG_WEIGHT_COMPOSITE +
    eligible["sharpe_score"] * SHARPE_WEIGHT_COMPOSITE
).round(2)

# Rank from best to worst
eligible = eligible.sort_values("composite_score", ascending=False).reset_index(drop=True)
eligible["rank"] = eligible.index + 1

print("Top 25 by composite score:")
eligible[[TICKER_COL, "ESG_score", "sharpe_ratio", "composite_score", "rank"]].head(25)

Top 25 by composite score:


,ticker,ESG_score,sharpe_ratio,composite_score,rank
0,ASML.AS,56.48,1.907,73.89,1
1,INGA.AS,55.04,1.655,67.73,2
2,HEXA-B.ST,59.20,1.506,67.10,3
3,ITX.MC,71.53,1.049,64.90,4
4,ULVR.L,66.06,1.194,64.66,5
5,AIR.PA,64.42,0.880,57.09,6
6,VOLV-B.ST,57.59,1.062,56.81,7
7,VOD.L,59.59,0.993,56.56,8
8,SAP.DE,67.86,0.720,55.79,9
9,CAP.PA,60.09,0.923,55.39,10


## Step 4 — Select final holdings and assign weights

We take the top 20 companies by composite score.  
Weights are proportional to composite score, capped at 10% per holding.

In [5]:
N_HOLDINGS = 20           # number of final holdings
MAX_WEIGHT = 0.10         # 10% cap per stock

portfolio = eligible.head(N_HOLDINGS).copy()

# Score-proportional weights
total_score = portfolio["composite_score"].sum()
portfolio["weight_raw"] = portfolio["composite_score"] / total_score

# Cap at MAX_WEIGHT and redistribute excess
capped = portfolio["weight_raw"].clip(upper=MAX_WEIGHT)
excess = portfolio["weight_raw"].sum() - capped.sum()
uncapped_mask = portfolio["weight_raw"] < MAX_WEIGHT
capped[uncapped_mask] += excess * (capped[uncapped_mask] / capped[uncapped_mask].sum())
portfolio["weight"] = (capped / capped.sum()).round(4)  # ensure exact sum to 1

print(f"Final portfolio: {len(portfolio)} holdings")
print(f"Weights sum to: {portfolio['weight'].sum():.4f} (should be 1.0)")
print(f"Max weight: {portfolio['weight'].max():.1%}")
print()
portfolio[[TICKER_COL, "ESG_score", "sharpe_ratio", "composite_score", "weight"]]

Final portfolio: 20 holdings
Weights sum to: 1.0000 (should be 1.0)
Max weight: 6.4%



,ticker,ESG_score,sharpe_ratio,composite_score,weight
0,ASML.AS,56.48,1.907,73.89,0.0641
1,INGA.AS,55.04,1.655,67.73,0.0588
2,HEXA-B.ST,59.20,1.506,67.10,0.0582
3,ITX.MC,71.53,1.049,64.90,0.0563
4,ULVR.L,66.06,1.194,64.66,0.0561
5,AIR.PA,64.42,0.880,57.09,0.0496
6,VOLV-B.ST,57.59,1.062,56.81,0.0493
7,VOD.L,59.59,0.993,56.56,0.0491
8,SAP.DE,67.86,0.720,55.79,0.0484
9,CAP.PA,60.09,0.923,55.39,0.0481


## Step 5 — Portfolio summary statistics

These are the numbers that go on the factsheet and in the presentation.

In [6]:
weighted_esg   = (portfolio["ESG_score"] * portfolio["weight"]).sum()
weighted_sharpe = (portfolio["sharpe_ratio"] * portfolio["weight"]).sum()

# Universe averages for comparison
universe_esg   = combined["ESG_score"].mean()
universe_sharpe = combined["sharpe_ratio"].mean()

print("=== PORTFOLIO vs UNIVERSE ===")
print(f"{'Metric':<25} {'Portfolio':>12} {'Universe':>12} {'Improvement':>12}")
print("-" * 65)
print(f"{'Weighted ESG Score':<25} {weighted_esg:>12.1f} {universe_esg:>12.1f} {weighted_esg - universe_esg:>+12.1f}")
print(f"{'Weighted Sharpe Ratio':<25} {weighted_sharpe:>12.3f} {universe_sharpe:>12.3f} {weighted_sharpe - universe_sharpe:>+12.3f}")
print(f"{'Number of Holdings':<25} {len(portfolio):>12} {len(combined):>12}")

=== PORTFOLIO vs UNIVERSE ===
Metric                       Portfolio     Universe  Improvement
-----------------------------------------------------------------
Weighted ESG Score                62.7         60.0         +2.7
Weighted Sharpe Ratio            0.986        0.423       +0.563
Number of Holdings                  20           70


## Step 6 — Save portfolio

In [7]:
today = str(date.today())
out_path = f"../outputs/portfolio/final_portfolio_{today}.csv"
portfolio.to_csv(out_path, index=False)
print(f"Portfolio saved: {out_path}")

# Also save universe scores for benchmark comparison
combined.to_csv(f"../outputs/portfolio/universe_scores_{today}.csv", index=False)
print(f"Universe scores saved.")

Portfolio saved: ../outputs/portfolio/final_portfolio_2026-05-06.csv
Universe scores saved.


## ✅ Notebook complete

You now have:
- **Final 20-stock portfolio** with weights that sum to 100%
- **Exclusion log** with reasons for every rejected company
- **Portfolio vs. universe comparison** (ESG improvement, Sharpe improvement)

**Next:** Open `05_reporting.ipynb` to generate the charts and factsheet.